# Understanding the Gather-and-Aggregate Mechanism in Language Models
This notebook explores how both Transformer and State-Space Model (SSM) architectures develop specialized mechanisms for in-context retrieval. We demonstrate that performance differences between these architectures stem primarily from a small number of critical neural network components rather than fundamental architectural limitations.

### ⚙️ The Gather-and-Aggregate Mechanism
The G&A mechanism is a collaborative process between two specialized heads across different layers that enables effective information retrieval in language models. This mechanism works through:

- Gather heads: identify and condense segments of input tokens into a final representation <br>
- Aggregate heads: process and extract the necessary information from the condensed representation

Our experiments provide compelling evidence that these specialized head pairs are fundamental to in-context learning capabilities. As we'll demonstrate, selectively disabling even a single head in these pairs can cause MMLU performance to collapse from state-of-the-art levels to random guessing.

For more details on the complete research, please refer to [the paper](https://arxiv.org/abs/2501.00000).


### 🚀 Approach 
**Note:** This notebook intentionally reverses the explanation order from the original paper to provide a different perspective on the findings:

1. First, we define Gather-and-Aggregate (G&A) heads and their mechanisms (Paper Section 4) <br>
2. Then, we demonstrate their critical importance through MMLU performance (Paper Section 3) <br>
3. Finally, we examine why SSM models struggle with these mechanisms (Paper Section 5)


### 📝 Contributing to the Notebook
🐛 Is there a bug? 💡 Do you have a suggestion for improvements?

Please open an issue or a pull request. We appreciate your feedback!


## Table of Contents

1. [Environment Setup](#env_setup)

2. [Models](#sec-models)

3. [Benchmarking](#sec-benchmarking)

    a. [MMLU Task](#subsec-mmlu)

    b. [Knowledge-Focused Tasks](#subsec-knowledge_focused)

4. [Gather-and-Aggregate Heads](#sec-gather_n_aggregate)

    a. [Gather Head](#subsec-gather)

    b. [Aggregate Head](#subsec-aggregate)

5. [Storing Knowledge vs. Retrieval in the Network](#sec-motivation)

    a. [MMLU Relies on a Single Head](#subsec-mmlu_reliance_single)

    b. [MMLU Relies on Two Crucial Heads](#subsec-mmlu_reliance_two)

    c. [Zooming In: MMLU Relies on Two Crucial Heads](#subsec-gather_n_aggregate)

    d. [Retrieval Emerges as a Key Factor in MMLU](#subsec-retrieval_key)

6. [Hybrid Experiments: SSM Struggles with Aggregation](#sec-ssm_struggles)

    a. [Hybrid Delegate](#subsec-hybrid-delegate)

    b. [Hybrid Replacements](#subsec-hybrid-replacements)


# 1️⃣ Environment Setup <a id="sec-env_setup"></a>
We begin by configuring the environment variables and importing necessary libraries.

In [ ]:
# %%bash
# pip install numpy==1.26.4 scipy==1.14.1 torch==2.4.0
# git clone https://github.com/cartesia-ai/edge.git
# pip install edge/cartesia-pytorch
# cp -r edge/cartesia-pytorch/cartesia_pytorch $(python -c "import site; print(site.getsitepackages()[0])")
# rm -rf edge
# pip install matplotlib==3.8.3 mamba-ssm==2.2.4
# pip install transformers==4.49.0 lm_eval==0.4.7
# pip install flash-attn==2.7.0.post2 wonderwords==2.2.0

In [ ]:
import os
import importlib.metadata
from functools import partial

required_versions = {
    "numpy": "1.26.4",
    "scipy": "1.14.1",
    "torch": "2.4.0",
    "transformers": "4.49.0",
    "lm_eval": "0.4.7",
    "flash-attn": "2.7.0.post2",
    "wonderwords": "2.2.0",
    "matplotlib": "3.8.3",
    "mamba-ssm": "2.2.4",
    "cartesia-pytorch": "0.0.1",
}

for pkg, required_version in required_versions.items():
    installed_version = importlib.metadata.version(pkg)
    assert installed_version == required_version, f"{pkg} version mismatch: Expected {required_version}, but found {installed_version}"

print("All package versions match!")

# 2️⃣ Models <a id="sec-models"></a>

Throughout this notebook, we will use the following models:

1. [**Llama-3.1-8B-Instruct**](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) (Transformer-based)

2. [**Falcon-Mamba-7B-Instruct**](https://huggingface.co/tiiuae/falcon-mamba-7b-instruct) (Mamba-1-based)

3. [**Llamba-8B-untied**](https://huggingface.co/cartesia-ai/Llamba-8B-untied) (Mamba-2-based distilled model from Llama)

4. [**Zamba-2-7B**](https://huggingface.co/Zyphra/Zamba2-7B) (Hybrid model with both attention and Mamba-2 layers)  

5. [**Llamba-8B-untied-unaligned**](https://huggingface.co/goombalab/Llamba-8B-untied-unaligned) (Mamba-2-based distilled model from Llama, without end-to-end alignment)

To load these models, we use the `get_model` function, which is defined in the `utils.py` file.
The `get_model` function loads the model from the Hugging Face model hub and returns the model and tokenizer,
along with the number of heads and the dimension of each head.
```python
model, tokenizer, num_heads, head_dim = get_model("llama") # or "falcon", "zamba", "llamba", "llamba_unaligned"
```

# 3️⃣ Benchmarking Functions <a id="sec-benchmarking"></a>
As described in Section 3 of the paper, we need to evaluate models on two distinct categories of tasks, which will allow us to distinguish between a model's knowledge capacity and its retrieval capabilities.
We will use the [**lm-eval-harness**](https://github.com/EleutherAI/lm-evaluation-harness) benchmarking library to evaluate the models on these tasks.

Specifically, we will have two evaluation functions:
- `eval_mmlu` measures the accuracy of models on **MMLU**, a task that requires the model to retrieve information from the context.

- `eval_knowledge` measures the accuracy of models on **Knowledge-Focused Tasks**, which primarily assess factual knowledge (ARC, PIQA, Winogrande, etc.) with minimal dependence on retrieval.


### MMLU <a id="subsec-mmlu"></a>
MMLU is a multiple-choice benchmark that requires both factual knowledge and algorithmic retrieval capabilities.

MMLU's format distinctively presents multiple-choice questions across 57 diverse subjects, requiring models to process lengthy contexts containing both questions and answer choices, then identify the correct option by outputting the corresponding letter label (typically A, B, C, D).
An example of an MMLU question is:
```
_______________ is the central node of 802.11 wireless operations.
A. WPA
B. Access Point
C. WAP
D. Access Port
Answer:
```
To evaluate this question, the model needs to understand the context and retrieve the correct letter corresponding to the answer (B).

In [ ]:
from utils import run_eval

def eval_mmlu(model, tokenizer):
    """
    Evaluate the model on the MMLU task
    """
    return run_eval(model, tokenizer, tasks=["mmlu"])['results']['mmlu']['acc,none']

### Knowledge-Focused Tasks <a id="subsec-knowledge_focused"></a>

Knowledge-Focused Tasks evaluate factual knowledge with minimal reliance on retrieval. 
This category includes Winogrande, OpenbookQA, PIQA, ARC Easy, ARC Challenge, and HellaSwag.

For example, an ARC Challenge question might be:
```
Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
Answer:
```
This question is accompanied by four possible answers: ```[dry palms, wet palms, palms covered with oil, palms covered with lotion]```.

To assess the model’s performance, **lm-eval-harness** appends all four answer choices to the question and determine which one receives the highest probability score. 
For instance, the model should assign the highest likelihood to the correct answer:
```
Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
Answer: dry palms
```


The key difference between these tasks and MMLU is in the prediction format. While MMLU requires models to output letter labels (A, B, C, D) after processing question-answer pairs, tasks like ARC, OpenBookQA, and HellaSwag expect models to generate the full text of the correct answer. 
Similarly, in PIQA and Winogrande, models must output the complete, more plausible statement. This shifts the evaluation focus from letter-based retrieval to direct knowledge application.

In [ ]:
from utils import run_eval

def eval_knowledge(model, tokenizer):
    """
    Evaluate the model on the knowledge tasks (Winogrande, OpenBookQA, PIQA, HellaSwag, ARC Challenge, ARC Easy)
    """
    knowledge_tasks = ["winogrande", "openbookqa", "piqa", "hellaswag", "arc_challenge", "arc_easy"]
    results = run_eval(model, tokenizer, tasks=knowledge_tasks)['results']
    return sum([results[task]['acc,none'] for task in knowledge_tasks]) / len(knowledge_tasks)

# 4️⃣ Gather-and-Aggregate Heads <a id="sec-gather_n_aggregate"></a>

We now explain how language models perform in-context retrieval through a Gather-and-Aggregate (G&A) mechanism, where two specialized heads collaborate:

- Gather Head: Identifies and condenses segments of input text into key summary tokens

- Aggregate Head: Processes these summary tokens in later layers to extract relevant information

Similarly to the paper, we will use the multiple-choice question about 802.11 wireless operations presented in the [MMLU task introduction](#subsec-mmlu) to illustrate the G&A mechanism.

<!-- Using an MMLU multiple-choice question about 802.11 wireless operations, the document illustrates how:

- The Gather Head isolates conceptual units (like answer choices) and condenses each segment into its final token,

- The Aggregate Head later processes these condensed representations to select the most relevant information. -->

### Gather Heads <a id="subsec-gather"></a>

A Gather head is a specialized attention mechanism that partitions the input context into logical segments and uses the last token of each segment as a representative summary. This representative token is transformed through a weighted linear combination of all elements in its segment. 
Specifically, the Gather head applies attention weights such that the last token "attends to" all preceding tokens within its segment, effectively absorbing their semantic information.

For example, in a multiple-choice question format, Llama-3.1-8B-Instruct gathers the tokens "A.", "WPA", and "\n" into the representative "\n" token. 
This mechanism transforms the computationally intensive task of analyzing numerous tokens into the more efficient task of processing a single summary vector per segment.

Throughout the rest of this notebook, we will use the gather heads we have already found (see Section 3 in the paper for more details). `GATHER_HEADS` is a dictionary that contains a mapping from model names to a tuple of `(layer_index, head_index)` for the gather head(s) in that model. For instance, Llama-3.1-8B-Instruct has a gather head in layer 16 at the 22nd head.

In [ ]:
GATHER_HEADS = {
    "llama": (16, [22]),
    "falcon": (35, [3186, 5143, 6607, 7305]),
    "llamba": (16, [10]),
}

### Aggregate Heads <a id="subsec-aggregate"></a>

The Aggregate Head operates in layers **following** a Gather Head and processes the summary tokens previously generated. 
It enables tokens later in the sequence to interact with all representative tokens, maintaining a global view of the context. 
By attending to these representative elements, it forms a linear combination of them, assigning greater weights to the most relevant information. 

In multiple-choice answers, the token position corresponding to "Answer:" combines information from all the newline tokens that represent each answer choice. This combination heavily weights the newline token from the correct answer choice (B in this case), performing what resembles an *argmax*-like operation.

Similar to the Gather Heads, we have already found the Aggregate Heads for the models we are using in this notebook. `AGGREGATE_HEADS` is a dictionary that contains a mapping from model names to a tuple of `(layer_index, head_index)` for the aggregate head(s) in that model. For instance, Llama-3.1-8B-Instruct has an aggregate head in layer 17 at the 24th head.

In [ ]:
AGGREGATE_HEADS = {
    "llama": (17, [24]),
    "falcon": (36, [1565, 1906, 3873, 4925]),
    "llamba": (17, [3, 9]),
}

### Plotting Gather-and-Aggregate Heads

For a better understanding of Gather-and-Aggregate heads, we plot the Gather-and-Aggregate heads for Llama-3.1-8B-Instruct and Llamba-8B models.

We see that while the Gather Head isolates the answer choices, the Aggregate Head processes these choices to select the most relevant information.
Moreover, Llamba-8B, an SSM, exhibit smoother attention patterns compared to Llama-8B, making it difficult to execute the sharp token shifts needed for G&A, requiring more of their instances to achieve comparable performance.

In [ ]:
from visualize import plot_attention
from utils import get_model

input_text = '_______________ is the central node of 802.11 wireless operations.\nA. WPA\nB. Access Point\nC. WAP\nD. Access Port\nAnswer:' # Answer is B

for model_name in ['llamba', 'llama']:
    model, tokenizer, num_heads, head_dim = get_model(model_name)
    gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]
    plot_attention(model, tokenizer, gather, aggregate, input_text)

# 5️⃣ Storing Knowledge vs. Retrieval in the Network <a id="sec-motivation"></a>

Transformer and SSM language models of similar sizes perform comparably on tasks requiring factual knowledge and basic linguistic understanding. However, their performance diverges significantly on skill-based tasks like memorization, retrieval, and precise copying. Retrieval has been identified as the primary factor behind the perplexity gap between attention-based models and SSM variants on the Pile dataset. Similarly, SSMs struggle more than Transformers on MMLU—a multiple-choice benchmark that requires algorithmic precision to extract the correct answer letter from context.
This section presents two key findings:

(a) The challenge of MMLU lies primarily in **retrieving** the correct letter from the context rather than extracting factual knowledge. <br>
(b) Both model types encode retrieval abilities in a small set of specialized heads, extending prior work that focused solely on Transformers.

Our analysis follows a systematic progression: <br>
1. [MMLU Relies on a Single Crucial Layer](#subsec-mmlu_reliance_single) shows that MMLU performance initially appears to depend on a specific critical layer rather than the network’s overall stored knowledge, based on cumulative layer pruning. <br>
2. [MMLU Relies on Two Crucial Layers](#subsec-mmlu_reliance_two) refines this finding through more targeted experiments, revealing that MMLU actually relies on the coordination between two essential layers, both of which must be present for successful performance. <br>
3. [Zooming In: MMLU Relies on Two Crucial Heads](#subsec-gather_n_aggregate) further refines this perspective by shifting from layers to individual heads, showing that MMLU disproportionately relies on a few specific heads rather than distributed knowledge. <br>
4. [Retrieval Emerges as a Key Factor in MMLU](#subsec-retrieval_key) shows that these heads are part of a retrieval mechanism that is consistently present across all architectures.


## MMLU Relies on a Single Crucial Layer <a id="subsec-mmlu_reliance_single"></a>

All architectures reveal a striking pattern: while performance on knowledge-focused benchmarks decreased uniformly as layers were pruned, MMLU performance remained remarkably stable until a specific layer was removed. 
At this point, MMLU performance dropped dramatically, suggesting this particular layer encodes crucial skills required for MMLU. 
This observation also raises an intriguing question about how these models can maintain high MMLU scores while apparently retaining only a fraction of their knowledge capacity.

<div style="text-align: center;">
  <img src="./assets/knowledge_vs_mmlu_llama.png" alt="(Not shown here; see Figure 2 in paper)" width="50%" style="display: block; margin: 0 auto;" />
  <p style="font-size: 90%; text-align: center; max-width: 55%; margin: 0.5em auto 0 auto;">
    <strong>Figure 2:</strong> 
    Performance of Llama-3.1-8B on knowledge tasks and MMLU as layers are gradually pruned.
  </p>
</div>

Key observations:<br>
1. MMLU scores stay stable despite declining knowledge tasks performance;<br>
2. a sharp MMLU drop identifies a critical skill-encoding layer;<br>
3. 30% of the model’s original score on knowledge tasks is sufficient to reach a state-of-the-art MMLU score.

Since recreating this figure requires extensive evaluation, we will only reproduce the transition point where MMLU performance collapses. 
We will use the [evaluation functions](#benchmarking-functions) defined earlier to evaluate the models on MMLU and knowledge-focused tasks after pruning the layers.


In [ ]:
from utils import get_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer = get_model(model_name)[0:2]
model.backbone.layers = model.backbone.layers[0:AGGREGATE_HEADS[model_name][0]+1]

print(f"Evaluating the model (up to and including the Gather and Aggregate layers):")
mmlu_score_0 = eval_mmlu(model, tokenizer)
knowledge_score_0 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_0}\nKnowledge score: {knowledge_score_0}")

print(f"Evaluating the model without the layer with the Aggregate head:")
layer = model.backbone.layers.pop(-1)
mmlu_score_1 = eval_mmlu(model, tokenizer)
knowledge_score_1 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_1}\nKnowledge score: {knowledge_score_1}")

print("-"*50)

print(f"Removing the Aggregate head caused the MMLU score to drop from {mmlu_score_0} to {mmlu_score_1}, \n \
      while the Knowledge score remained the same ({knowledge_score_0} to {knowledge_score_1}).")


## MMLU Relies on Two Crucial Layers <a id="subsec-mmlu_reliance_two"></a>

Based on insights from the [previous section](#subsec-mmlu_reliance_single), we refer to the \textit{minimal model} as the smallest subnetwork that retains all layers up to (and including) the layer before the sharp drop in MMLU performance. 
We now repeat the previous experiment, this time *on the minimal model*, with a slight modification. 
Instead of cumulatively removing one layer at a time from the end of the network, we remove a single layer, evaluate the model performance without it, and then restore the layer before proceeding to the next one.

We define 'get_minimal_model' function to get the minimal model by removing all layers after the critical G&A heads.
```python
def get_minimal_model(model_name, layer_idx):
    model, tokenizer, num_heads, head_dim = get_model(model_name)
    model.backbone.layers = model.backbone.layers[0:layer_idx+1]
    return model, tokenizer, num_heads, head_dim
```

We now show that **the last two layers** show no correlation between MMLU and knowledge tasks. 
Pruning either significantly reduces MMLU accuracy while leaving knowledge tasks unaffected, highlighting their essential role.

<div style="text-align: center;">
  <img src="assets/knowledge_vs_mmlu_minimal.png" alt="Not shown here; see Figure 3 in paper" width="50%"/>
  <p style="font-size: 90%; text-align: left; display: inline-block; max-width: 50%;">
    <strong>Figure 3:</strong> Performance of Llama-3.1-8B-Minimal with layers removed and restored. We remove one layer at a time, evaluate performance on MMLU and knowledge tasks’ score, and then restore the layer before proceeding to the next.
  </p>
</div>

The results on Llama-3.1-8B-Minimal reveal that both layers 16 and 17 are critical for MMLU performance, as pruning either of the last two layers of the minimal model causes a significant drop on MMLU scores while leaving knowledge tasks’ score largely unaffected.

Similarly to the previous section, recreating this figure requires extensive evaluation, thus we will only reproduce the points where knowledge-focused tasks' performance remains stable while MMLU performance collapses.

In [ ]:
from utils import get_minimal_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_minimal_model(model_name, AGGREGATE_HEADS[model_name][0])

print(f"Evaluating the Minimal model without the Gather head:")
model.backbone.layers.pop(-2) # Remove the layer with the Gather head while keeping the layer with the Aggregate head
mmlu_score_2 = eval_mmlu(model, tokenizer)
knowledge_score_2 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_2}\nKnowledge score: {knowledge_score_2}")

print("-"*50)
print(f"Removing the Gather head caused the MMLU score to drop from {mmlu_score_0} to {mmlu_score_2}, \n \
      while the Knowledge score remained the same ({knowledge_score_0} to {knowledge_score_2}).")

## Zooming In: MMLU Relies on Two Crucial Heads <a id="subsec-gather_n_aggregate"></a>
This section includes functions to manipulate specific **attention heads** in the model to test their contribution to retrieval.  
- `keep_heads` selectively retains specific heads in a layer while nullifying others. <br><br>
- `remove_heads` disables selected heads to analyze their impact. 

The experiments here align with **Table 1 in Section 3.3 (Zooming In: MMLU Relies on Two Crucial Heads)**, where the importance of Gather and Aggregate heads was demonstrated.

To further validate the importance of these identified critical heads, we removed all noncritical heads from the last two layers and evaluated performance across four configurations:

(1) no critical heads retained, <br><br>
(2) only critical heads from the last layer retained, <br><br>
(3) only critical heads from the second last layer retained, and <br><br>
(4) critical heads in both layers retained.

### 1. Only critical heads from the last layer retained
This experiment isolates only the **Aggregate head** while nullifying all other heads in both the last and second-to-last layers. 

The Aggregate head is responsible for combining summarized representations from earlier layers and selecting the most relevant information.

In [ ]:
from utils import keep_heads, remove_heads
from utils import get_minimal_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_minimal_model(model_name, AGGREGATE_HEADS[model_name][0])
gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]

keep_heads = partial(keep_heads, num_heads=num_heads, head_dim=head_dim)
remove_heads = partial(remove_heads, num_heads=num_heads, head_dim=head_dim)

keep_heads(model, layer_idx=gather[0], heads=gather[1])
keep_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
remove_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
mmlu_score_3 = eval_mmlu(model, tokenizer)
knowledge_score_3 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_3}\nKnowledge score: {knowledge_score_3}")
print("-"*50)
print(f"Removing the Aggregate heads caused the MMLU score to drop from {mmlu_score_0} to {mmlu_score_3}, \n \
      while the Knowledge score remained the same ({knowledge_score_0} to {knowledge_score_3}).")

### 2. Only critical heads from the second last layer retained
This experiment isolates only the **Gather head**, removing all other heads in both layers. 

The Gather head identifies key segments of text and summarizes them into concise representations.

In [ ]:
from utils import keep_heads, remove_heads
from utils import get_minimal_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_minimal_model(model_name, AGGREGATE_HEADS[model_name][0])
gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]

keep_heads = partial(keep_heads, num_heads=num_heads, head_dim=head_dim)
remove_heads = partial(remove_heads, num_heads=num_heads, head_dim=head_dim)

keep_heads(model, layer_idx=gather[0], heads=gather[1])
keep_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
remove_heads(model, layer_idx=gather[0], heads=gather[1])
mmlu_score_4 = eval_mmlu(model, tokenizer)
knowledge_score_4 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_4}\nKnowledge score: {knowledge_score_4}")
print("-"*50)
print(f"Removing the Gather heads caused the MMLU score to drop from {mmlu_score_0} to {mmlu_score_4}, \n \
      while the Knowledge score remained the same ({knowledge_score_0} to {knowledge_score_4}).")

### 3. Critical heads in both layers retained.
In this experiment, **both the Gather and Aggregate heads**, removing all other heads in both layers. 

This configuration ensures that the minimal components necessary for retrieval remain intact.

In [ ]:
from utils import keep_heads
from utils import get_minimal_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_minimal_model(model_name, AGGREGATE_HEADS[model_name][0])
gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]

keep_heads = partial(keep_heads, num_heads=num_heads, head_dim=head_dim)

keep_heads(model, layer_idx=gather[0], heads=gather[1])
keep_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
mmlu_score_5 = eval_mmlu(model, tokenizer)
knowledge_score_5 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_5}\nKnowledge score: {knowledge_score_5}")
print("-"*50)
print(f"Keeping only the Gather and Aggregate heads kept the MMLU score at {mmlu_score_5}/{mmlu_score_0}, \n \
      and the Knowledge score at {knowledge_score_5}/{knowledge_score_0}.")

### 4. No critical heads retained.
This configuration removes both the Gather and Aggregate heads, leaving only noncritical heads in the last two layers.

In [ ]:
from utils import remove_heads
from utils import get_minimal_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_minimal_model(model_name, AGGREGATE_HEADS[model_name][0])
gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]

remove_heads = partial(remove_heads, num_heads=num_heads, head_dim=head_dim)

remove_heads(model, layer_idx=gather[0], heads=gather[1])
remove_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
mmlu_score_6 = eval_mmlu(model, tokenizer)
knowledge_score_6 = eval_knowledge(model, tokenizer)
print(f"MMLU score: {mmlu_score_6}\nKnowledge score: {knowledge_score_6}")
print("-"*50)
print(f"Removing both the Gather and Aggregate heads caused the MMLU score to drop to {mmlu_score_6}/{mmlu_score_0}, \n \
        and the Knowledge score at {knowledge_score_6}/{knowledge_score_0}.")


## Retrieval Emerges as a Key Factor in MMLU <a id="subsec-retrieval_key"></a>

Our earlier findings demonstrated that two attention heads from distinct layers collaborate to effectively solve MMLU across all tested architectures. We now present evidence that these heads constitute part of a broader *in-context retrieval mechanism*. To establish this retrieval function, we analyze how removing one critical head impacts performance on a synthetic key-value (KV) retrieval task.

We conducted experiments across 55 KV-Retrieval configurations with progressively increasing numbers of key-value pairs, each comprising 1,000 synthetic samples.
For each configuration, we evaluated Llama-3.1-8B twice: once as-is and once with *L17H24* (the head previously identified as critical in MMLU experiments) removed. We then compared the performance of the two configurations on both MMLU and the KV-Retrieval task.

<div style="text-align: center;">
  <img src="assets/kv_retrieval_llama_8b.png" alt="Not shown here; see Figure 4 in paper" width="50%"/>
  <p style="font-size: 90%; text-align: left; display: inline-block; max-width: 50%;">
    <strong>Figure 4:</strong> Evaluation of Llama-3.1-8B on KV-Retrieval task with and without L17H24 (the head previously identified in MMLU experiments).
  </p>
</div>

This confirms that the G&A mechanism is not specific to MMLU, but represents a general retrieval capability that emerges in language models during pretraining.
The code below reproduces the key-value retrieval experiment for each model.
As shown in Figure 4, removing this head causes a substantial decline in retrieval accuracy, with the performance gap widening as the number of key-value pairs increases.

In [ ]:
from kv_retrieval import MemorizationDataset, eval_kv
from utils import remove_heads, get_model

model_name = 'llama' # or 'llamba' or 'falcon'
model, tokenizer, num_heads, head_dim = get_model(model_name)
gather, aggregate = GATHER_HEADS[model_name], AGGREGATE_HEADS[model_name]
remove_heads = partial(remove_heads, num_heads=num_heads, head_dim=head_dim)

datasets = {num_pairs: list(MemorizationDataset(num_pairs=num_pairs, num_examples=1000)) for num_pairs in range(10, 55, 5)}

print("num_pairs;accuracy")
results_0 = []
for num_pairs, dataset in datasets.items():
    results_0.append(eval_kv(model, tokenizer, dataset)[1])
    print(f"{num_pairs};{results_0[-1]:.3f}")
print("-"*50)

print("num_pairs;accuracy")
results_1 = []
remove_heads(model, layer_idx=aggregate[0], heads=aggregate[1])
for num_pairs, dataset in datasets.items():
    results_1.append(eval_kv(model, tokenizer, dataset)[1])
    print(f"{num_pairs};{results_1[-1]:.3f}")

print("-"*50)
print("Overall, removing the Aggregate heads caused the KV retrieval score to drop:")
print("num_pairs;drop")
for i, num_pairs in enumerate(datasets.keys()):
    print(f"{num_pairs};{results_0[i] - results_1[i]:.3f}")
print("Interestingly, the more pairs the dataset contains, the larger the gap between the scores with and without the Aggregate heads.\n \
      This suggests that the Aggregate heads are more important as the manipulation complexity increases.")

# 6️⃣ Hybrid Experiments: SSM Struggles with Aggregation <a id="sec-ssm_struggles"></a>


This section explores the **role of hybrid models**, which combine attention and SSM layers, and is part of the broader discussion in **SSM struggles with Aggregation (Section 5)**, which provides evidence that the known weakness of a single SSM head in retrieval is manifested as a limitation in aggregation tasks.

Specifically, we will run two experiments:
1. **Hybrid Delegate**: This experiment demonstrates that hybrid models naturally assign retrieval-related functions to attention-based components, as shown in **Section 5.3** of the paper. <br><br>
2. **Hybrid Replacements**: This experiment replaces layers in an **unaligned Llamba-8B model** with layers from a **Transformer-based Llama model** to see if performance improves, as shown in **Section 5.4** of the paper.

## Hybrid Models Delegate Aggregation to Attention <a id="subsec-hybrid-delegate"></a>

Our experiments demonstrate that hybrid models naturally assign retrieval-related functions to attention-based components. By masking the last row of every attention matrix (effectively disabling all potential Aggregate heads in the attention layers), we observe a dramatic drop in MMLU score from 64.3% to 34.9%, while knowledge task performance remains stable.

This not only confirms that hybrid models assign Aggregate head functionality to attention layers, but also reveals SSMs’ fundamental weakness in aggregation tasks: during pretraining, SSMs struggle to develop effective Gather-and-Aggregate mechanisms.

In [ ]:
from utils import get_model

model, tokenizer, num_heads, head_dim = get_model('zamba')

model.mask_last_row = True
mmlu_score_7 = eval_mmlu(model, tokenizer)
knowledge_score_7 = eval_knowledge(model, tokenizer)
print(f"Masking the last row of the attention matrix:\nMMLU score: {mmlu_score_7}, Knowledge score: {knowledge_score_7}")

model.mask_last_row = False
mmlu_score_8 = eval_mmlu(model, tokenizer)
knowledge_score_8 = eval_knowledge(model, tokenizer)
print(f"Unmasking the last row of the attention matrix:\nMMLU score: {mmlu_score_8}, Knowledge score: {knowledge_score_8}")

print("-"*50)
print(f"Masking the last row of the attention matrix caused the MMLU score to drop from {mmlu_score_8} to {mmlu_score_7}, \
       while the Knowledge score remained the same ({knowledge_score_8} to {knowledge_score_7}).\n \
       Since the last row of all Mamba layers (which predominantly compose the network) remains intact, \
       it means that most aggregation is done at the attention heads.")

## Hybrid Replacements <a id="subsec-hybrid-replacements"></a>

Our final experiment systematically replaces each layer in the Llamba-8B model with its corresponding layer from Llama-3.1-8B-Instruct to measure the impact on MMLU performance.

The results show that most layer replacements have little or even negative impact. However, replacing layer 17, which contains the Aggregate head, significantly improves MMLU performance from 33% to 50%.

<div style="text-align: center;">
  <img src="assets/hybrid_replacement.png" alt="Not shown here; see Figure 7 in paper" width="50%"/>
  <p style="font-size: 90%; text-align: left; display: inline-block; max-width: 50%;">
    <strong>Figure 7:</strong> 
    Results of the hybrid replacement experiment.
    Each Llamba-8B layer was replaced with its Llama-3.1-8B counterpart and tested on MMLU.
  </p>
</div>

This provides further understanding into potentially optimal placement of Gather-and-Aggregate heads in the network. Furthermore, it demonstrates that the SSM architecture struggles specifically with implementing the Aggregate head functionality, as evidenced by the significant performance improvement when replaced with a Transformer-based layer.


In [ ]:
from utils import get_model

model, tokenizer, num_heads, head_dim = get_model('unaligned_llamba', is_minimal=False)
aux_model, tokenizer, num_heads, head_dim = get_model('llama', is_minimal=False)
model.backbone.rotary_emb = aux_model.backbone.rotary_emb # support for rotary embeddings in hybrid setup

print(f"The MMLU score of the Unaligned LLamba model is: {eval_mmlu(model, tokenizer)}")

print("Hybrid Replacement:")
n_layers = len(model.backbone.layers)
for layer_idx in range(n_layers):
    layer = model.backbone.layers[layer_idx]
    model.backbone.layers[layer_idx] = aux_model.backbone.layers[layer_idx]
    if layer_idx == AGGREGATE_HEADS['llamba'][0]:
        print(f"Layer {layer_idx}/{n_layers} MMLU score: {eval_mmlu(model, tokenizer)} <-- This should yield the highest MMLU improvement")
    else:
        print(f"Layer {layer_idx}/{n_layers} MMLU score: {eval_mmlu(model, tokenizer)}")
    model.backbone.layers[layer_idx] = layer